# 🤖 LLM Fundamentals for Business Managers
### Essential Knowledge Every Leader Should Have (15-20 minutes)

---
## ⚙️ Setup - Run This First

In [13]:
!pip install litellm tiktoken python-dotenv ipython litellm -q

import os
from dotenv import load_dotenv
from litellm import completion
import tiktoken
from IPython.display import Markdown, display, HTML
import litellm

load_dotenv()

# Styling
NAVY = "#0B1F3A"
ICE = "#EAF2FF"
GOLD = "#B08D57"

def banner(text, color=NAVY):
    display(HTML(f"""
    <div style="background:{color};color:{ICE};padding:14px 22px;border-radius:8px;
                font-family:'Segoe UI',sans-serif;font-size:19px;font-weight:600;
                margin:14px 0 6px 0;letter-spacing:0.2px;">
        {text}
    </div>
    """))

def insight(text):
    display(HTML(f"""
    <div style="background:{ICE};border-left:5px solid {GOLD};padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:{NAVY};margin:10px 0 22px 0;">
        <b>💡 Business Insight:</b> {text}
    </div>
    """))

def warning_box(text):
    display(HTML(f"""
    <div style="background:#FBEAEA;border-left:5px solid #B23B3B;padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:#5A1A1A;margin:10px 0 22px 0;">
        <b>⚠️ Critical Warning:</b> {text}
    </div>
    """))

print("✅ Setup complete. You are ready to go!")

✅ Setup complete. You are ready to go!



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


---
## 1. What Are LLMs Really Doing?

Large Language Models are **advanced next-word predictors**. 
They don’t “understand” — they predict the most likely next token based on patterns learned during training.

---
## 2. Tokens — The Real Unit of Cost

In [7]:
enc = tiktoken.encoding_for_model("gpt-4o")

examples = {
    "One-line status update": "Backup job 4471 completed successfully at 2:03 AM.",
    "Short incident note": "Nightly backup for East Region cluster failed due to network timeout.",
    "Full postmortem excerpt": "At 01:47 AM, the scheduled backup for the East Region storage cluster failed..."
}

banner("Token Cost by Document Type")

for label, text in examples.items():
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"{label:<35} {words:>3} words → {tokens:>3} tokens")

insight("You are billed **per token**, not per word. A 10-page document (~5,000 tokens) summarized with gpt-4o-mini costs less than a penny.")

One-line status update                8 words →  14 tokens
Short incident note                  11 words →  13 tokens
Full postmortem excerpt              13 words →  18 tokens


---
## 3. Temperature: Predictable vs Creative

In [8]:
prompt = "Draft a one-paragraph internal summary for leadership about a recurring backup failure on the East Region cluster."

for temp in [0.0, 0.7, 1.4]:
    banner(f"Temperature = {temp} {'(Strict & Consistent)' if temp == 0.0 else '(Balanced)' if temp == 0.7 else '(Creative & Exploratory)'}", color=GOLD if temp > 0.5 else NAVY)
    resp = completion(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=temp)
    display(Markdown(resp.choices[0].message.content.strip()))
    print("─" * 80)

insight("**Low temperature (0.0-0.3)** → Best for finance, legal, compliance, reports.<br>**Higher temperature (0.7-1.0)** → Best for brainstorming, marketing, creative content.")

Subject: Summary of Recurring Backup Failure on East Region Cluster

Dear Leadership Team,

We are currently experiencing a recurring backup failure on the East Region cluster, which has been impacting our data integrity and recovery processes. The issue was first identified on [insert date], and subsequent investigations have revealed that the failures are primarily due to [insert identified cause, e.g., network latency, hardware malfunctions, or software bugs]. Our IT team is actively working with [insert relevant vendors or internal teams] to diagnose and resolve the root cause. In the interim, we have implemented temporary measures to mitigate data loss risks, including [insert temporary solutions, e.g., increasing backup frequency or utilizing alternative storage solutions]. We are prioritizing this issue and will provide regular updates on our progress. Your understanding and support during this time are greatly appreciated as we work towards a permanent resolution.

Best regards,

[Your Name]
[Your Position]

────────────────────────────────────────────────────────────────────────────────


Subject: Summary of Recurring Backup Failure on East Region Cluster

Dear Leadership Team,

We are currently experiencing a recurring issue with the backup processes on our East Region cluster, leading to consistent failures over the past two weeks. The initial investigation suggests that the problem may be linked to a configuration error introduced during a recent system update, which has disrupted the synchronization between our primary storage and backup servers. This issue has resulted in incomplete data backups, posing a potential risk to our data integrity and business continuity. Our IT team is actively working to resolve the configuration error and is collaborating with our software vendor to implement a patch. In the interim, we are conducting manual backups to ensure data is securely stored. We anticipate a resolution within the next 48 hours and will keep you updated on our progress. Your understanding and support during this process are greatly appreciated.

Best regards,

[Your Name]
[Your Position]

────────────────────────────────────────────────────────────────────────────────


Subject: Summary of Recurring Backup Failure on East Region Cluster

As of the most recent assessment, there has been a recurring issue affecting the backup operations within the East Region cluster, which persists despite previous mitigation efforts. This problem, identified on [specific date], has resulted in incomplete and failed backups, thus potentially jeopardizing data integrity and business continuity. The primary causes seem to be linked to network bandwidth limitations and irregularities in sync processes during peak hours. Additionally, a series of compiler warnings and errors related to software version mismatches have been documented. Immediate remedial steps are needed, including a review of allocation of resources during backup windows and a comprehensive audit of the system upgrade procedures to identify underlying roadblocks. Recommendations for remediation involve coordinating with both network infrastructure teams to optimize required bandwidth and collaboration with the software management team to ensure version compatibility. Continuous updates will be communicated on further diagnostics and resolutions enacted.

────────────────────────────────────────────────────────────────────────────────


---
## 4. ⚠️ The Hallucination Warning

In [9]:
warning_box("LLMs can generate fluent, confident, but completely false information.<br><br>"
            "<b>Real business risks:</b> Made-up policies, fake court cases, incorrect numbers, non-existent features.<br><br>"
            "<b>Rule:</b> Always verify critical outputs with human oversight.")

---
## 5. System Prompts — Turn One Assistant into Many Experts

In [10]:
def ask(system_prompt, user_prompt, model="gpt-4o-mini", temperature=0.0):
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature
    )
    return resp.choices[0].message.content.strip()

ticket = "Customer's nightly backup failed three times due to storage quota limit. Resolved by increasing capacity."

banner("Internal Technical Summarizer")
internal = "You are a concise IT support analyst. Be technical, factual, and structured."
display(Markdown(ask(internal, f"Summarize: {ticket}")))

banner("Customer-Facing Communicator", color=GOLD)
customer = "You are a warm, reassuring customer success manager. Avoid blame and use simple language."
display(Markdown(ask(customer, f"Write a short email to the customer about: {ticket}")))

insight("A good **system prompt** is like giving a new hire a clear job description. It dramatically changes tone, depth, and focus.")

**Issue Summary:**
- **Problem:** Nightly backup failed three times.
- **Cause:** Storage quota limit exceeded.
- **Resolution:** Increased storage capacity to resolve the issue.

Subject: Update on Your Nightly Backup

Hi [Customer's Name],

I hope this message finds you well! I wanted to let you know that we noticed your nightly backup had failed three times due to reaching the storage quota limit. 

Good news! We’ve increased your storage capacity, so your backups should run smoothly from now on. If you have any questions or need further assistance, please don’t hesitate to reach out.

Thank you for your understanding!

Best,  
[Your Name]  
Customer Success Manager

---
## 6. Same Question, Different Models + Real Cost Tracking

In [14]:
total_cost = 0.0

def compare_models(prompt):
    global total_cost
    models = ["gpt-4o", "anthropic/claude-haiku-4-5"]
    
    for model in models:
        try:
            resp = completion(model=model, messages=[{"role": "user", "content": prompt}], temperature=0.0)
            
            prompt_tokens = getattr(resp.usage, 'prompt_tokens', 0)
            completion_tokens = getattr(resp.usage, 'completion_tokens', 0)
            cost = litellm.completion_cost(resp)
            total_cost += cost
            
            banner(f"Model: {model}", color=GOLD)
            print(f"Input tokens : {prompt_tokens}")
            print(f"Output tokens: {completion_tokens}")
            print(f"This call    : ${cost:.6f}")
            print(f"Running total: ${total_cost:.6f}")
            display(Markdown(resp.choices[0].message.content.strip()))
            print("─" * 80)
            
        except Exception as e:
            print(f"Error with {model}: {e}")

compare_models("Summarize the key benefits and risks of using generative AI in our company.")

Input tokens : 24
Output tokens: 488
This call    : $0.004940
Running total: $0.004940


Using generative AI in your company can offer several key benefits, but it also comes with certain risks. Here's a summary of both:

### Benefits:

1. **Increased Efficiency and Productivity:**
   - Automates repetitive tasks, freeing up employees to focus on more strategic activities.
   - Speeds up content creation, data analysis, and other processes, leading to faster decision-making.

2. **Enhanced Creativity and Innovation:**
   - Generates new ideas, designs, and solutions that might not be immediately apparent to human teams.
   - Facilitates brainstorming and prototyping by providing diverse options and perspectives.

3. **Personalization and Customer Engagement:**
   - Creates personalized content and recommendations, improving customer experience and satisfaction.
   - Enhances marketing strategies by tailoring messages to specific audience segments.

4. **Cost Savings:**
   - Reduces the need for extensive human resources in certain areas, lowering operational costs.
   - Minimizes errors and waste, leading to more efficient resource utilization.

5. **Scalability:**
   - Easily scales operations without a proportional increase in costs, allowing for rapid growth and adaptation to market changes.

### Risks:

1. **Quality and Accuracy Concerns:**
   - May produce outputs that are incorrect, biased, or misleading, requiring careful oversight and validation.
   - Dependence on AI-generated content can lead to a lack of critical human oversight.

2. **Ethical and Legal Issues:**
   - Raises concerns about intellectual property rights, especially when generating content similar to existing works.
   - Potential for misuse in creating deceptive or harmful content, leading to reputational damage.

3. **Data Privacy and Security:**
   - Requires access to large datasets, which can pose risks to data privacy and security if not managed properly.
   - Vulnerable to cyberattacks that could compromise sensitive information.

4. **Job Displacement:**
   - Automation of tasks may lead to job displacement or require significant workforce retraining and upskilling.
   - Can create resistance or fear among employees, affecting morale and company culture.

5. **Dependence on Technology:**
   - Over-reliance on AI systems can lead to vulnerabilities if the technology fails or becomes obsolete.
   - Requires ongoing investment in technology and expertise to maintain and update AI systems.

Balancing these benefits and risks involves implementing robust governance frameworks, ensuring transparency, and fostering a culture of continuous learning and adaptation.

────────────────────────────────────────────────────────────────────────────────


Input tokens : 24
Output tokens: 302
This call    : $0.001534
Running total: $0.006474


# Generative AI: Key Benefits and Risks

## Benefits

**Efficiency & Productivity**
- Automate routine tasks (writing, coding, analysis)
- Faster content creation and iteration
- Reduced time on repetitive work

**Cost Reduction**
- Lower labor costs for certain functions
- Fewer resources needed for some processes

**Enhanced Decision-Making**
- Quick data analysis and insights
- Pattern recognition across large datasets

**Innovation**
- New product/service possibilities
- Competitive advantage if deployed strategically

## Risks

**Quality & Accuracy**
- Hallucinations and factual errors
- Requires human verification (adds back time)

**Security & Privacy**
- Data exposure if sensitive info enters AI systems
- Intellectual property concerns

**Compliance Issues**
- Copyright/licensing questions
- Regulatory uncertainty (varies by industry/region)

**Operational Risks**
- Over-reliance on AI outputs
- Skill atrophy in teams
- Integration challenges with existing systems

**Reputational**
- Customer/stakeholder concerns about authenticity
- Bias in outputs affecting brand

## Recommendation

Start with **low-risk pilots** in specific functions, establish clear governance (what data can be used, human review requirements), and build internal expertise before scaling.

What specific use cases are you considering?

────────────────────────────────────────────────────────────────────────────────


---
## Final Takeaways for Managers

1. **LLMs predict tokens**, they don’t retrieve facts.
2. **Tokens = money** — longer documents cost more.
3. Use **temperature** deliberately: low for precision, higher for creativity.
4. **Hallucinations are inevitable** — always verify critical content.
5. **System prompts** are your most powerful control tool.
6. Different models = different strengths and costs.
7. Combine **great prompting + human oversight** for best results.

**You don’t need to become a prompt engineer — you just need to know what to watch for.**